# Continuous control: DDPG, TD3, and SAC

_Reinforcement Learning_

**When actions are real-valued torques or steering angles, you cannot max over infinitely many actions — so learn a policy that outputs the best action directly, trained off-policy against a Q-critic.**

In a discrete action space you act by reading off $\arg\max_a Q(s,a)$ &mdash; cheap, because you can enumerate actions.
       In a continuous space that $\max$ is an optimization over infinitely many actions at every step. The escape:
       train a second network, the actor, to output the maximizing action directly.

       So you keep two networks. The critic $Q_\phi(s,a)$ judges how good a state-action pair is (just like in
       mod-actor-critic). The actor $\mu_\theta(s)$ proposes the action it
       believes is best. The actor is trained to make the critic happy: push $\theta$ so that $Q_\phi(s,\mu_\theta(s))$ goes up.
       That single sentence is the deterministic policy gradient.

---

This notebook is a practice scaffold for the **Continuous control: DDPG, TD3, and SAC** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium stable-baselines3
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — gymnasium + stable-baselines3 (PyTorch)

### Step 1 — Train SAC on Pendulum

Pendulum has a single continuous torque action in $[-2, 2]$, so we cannot enumerate actions and take an $\arg\max$. Soft Actor-Critic (SAC) is the modern default here: it is off-policy (it reuses experience from a replay buffer) and trains a stochastic actor against a Q-critic. We let it learn for 20k environment steps, which solves Pendulum in a few minutes. DDPG and TD3 share the exact same API — just swap the import.

In [ ]:
# Colab:  !pip install "stable-baselines3[extra]" gymnasium
import gymnasium as gym
from stable_baselines3 import SAC          # swap for DDPG or TD3 — same API, all off-policy

env = gym.make("Pendulum-v1")              # 1-D continuous torque action in [-2, 2]

# Soft Actor-Critic: the modern default for continuous control.
model = SAC("MlpPolicy", env, verbose=1, learning_rate=3e-4,
            buffer_size=100_000,           # replay buffer (off-policy, reuses experience)
            batch_size=256, gamma=0.99, tau=0.005)   # tau = Polyak target-net averaging
model.learn(total_timesteps=20_000)        # ~a few minutes; SAC solves Pendulum quickly

# --- DDPG / TD3 note: identical usage, just change the import ---
# from stable_baselines3 import TD3        # twin critics + target smoothing + delayed actor
# from stable_baselines3 import DDPG       # the original; needs added action noise to explore
# model = TD3("MlpPolicy", env, action_noise=...)   # see sb3 docs for NormalActionNoise

### Step 2 — Roll out the trained policy

To evaluate, we ask the actor for its action directly with `deterministic=True` — no $\arg\max$ search, the network simply outputs the action it believes is best. We run one episode of up to 200 steps and sum the rewards. A return near `0` means the pendulum is balanced upright (solved); a random policy scores roughly `-1200`.

In [ ]:
# Evaluate the trained policy.
obs, _ = env.reset(seed=0)
total = 0.0
for _ in range(200):
    action, _ = model.predict(obs, deterministic=True)   # actor outputs the action directly
    obs, reward, term, trunc, _ = env.step(action)
    total += reward
    if term or trunc:
        break
print("episode return:", round(total, 1))   # near 0 is solved; random is ~ -1200

### Step 3 — Peek under the hood: actor and twin critics

This is the PyTorch skeleton that libraries like SB3 implement for you. The **actor** $\mu_\theta(s)$ maps a state to an action, squashed through `tanh` so it stays inside the bounds. The **twin critics** $Q_{\phi_1}, Q_{\phi_2}$ each score a state-action pair; TD3 and SAC take the *minimum* of the two to fight Q-value over-estimation. The comments at the bottom spell out the deterministic policy gradient and the TD3/SAC target.

In [ ]:
# Under the hood (PyTorch sketch): a deterministic tanh actor + TWIN critics.
import torch
import torch.nn as nn

def mlp(inp, out, hidden=256):               # small 2-hidden-layer ReLU network
    return nn.Sequential(nn.Linear(inp, hidden), nn.ReLU(),
                         nn.Linear(hidden, hidden), nn.ReLU(),
                         nn.Linear(hidden, out))

class Actor(nn.Module):                      # mu_theta(s): state -> action in [-A, A]
    def __init__(self, s_dim, a_dim, a_max):
        super().__init__()
        self.net = mlp(s_dim, a_dim)
        self.a_max = a_max
    def forward(self, s):
        return self.a_max * torch.tanh(self.net(s))   # tanh squash keeps actions bounded

class TwinCritic(nn.Module):                 # Q_phi1, Q_phi2 — TD3/SAC use min(Q1,Q2)
    def __init__(self, s_dim, a_dim):
        super().__init__()
        self.q1 = mlp(s_dim + a_dim, 1)
        self.q2 = mlp(s_dim + a_dim, 1)
    def forward(self, s, a):
        sa = torch.cat([s, a], dim=-1)
        return self.q1(sa), self.q2(sa)

# Deterministic policy gradient: maximize Q1(s, mu(s)) -> actor_loss = -Q1(s, mu(s)).mean()
# Critic target (TD3):  y = r + gamma * min(Q1', Q2')(s', mu'(s') + clipped_noise)
# SAC adds an entropy bonus -alpha*log pi to that target and uses a stochastic actor.

## Visualize the data & results

_Off-policy methods reuse experience, so they should learn faster per environment step. How much return does each algorithm reach at a fixed sample budget on a continuous-control task — DDPG vs TD3 vs SAC vs PPO?_

### Step 1 — Model each algorithm's learning curve

Rather than run four full training jobs, we use a simple analytic proxy for sample efficiency: each algorithm's return after `steps` is modeled as `ceiling * (1 - exp(-rate * steps))`. The `rate` encodes how fast it learns per sample and `ceiling` its asymptotic performance. The chosen numbers reflect the well-known qualitative ordering on continuous control.

In [ ]:
# Illustrative sample-efficiency proxy (NO gym): model each algorithm's learning curve as
# return(steps) = ceiling * (1 - exp(-rate * steps)), evaluated at a fixed budget.
# 'rate' encodes per-sample learning speed; 'ceiling' the asymptotic performance.
# Values chosen to reflect the standard qualitative ordering on continuous control.
budget = 50_000
algos = {
    # name:  (ceiling, rate per step)
    "DDPG": (300.0, 0.000010),   # off-policy but over-estimates -> brittle, lower & slower
    "TD3":  (340.0, 0.000026),   # twin critics fix DDPG -> fast & high
    "SAC":  (360.0, 0.000034),   # max-entropy -> fastest & most stable here
    "PPO":  (330.0, 0.000013),   # on-policy: robust but discards samples -> slower per step
}

### Step 2 — Evaluate returns at a fixed sample budget

We plug the fixed budget into each algorithm's curve and print the resulting return. The ordering SAC > TD3 > PPO > DDPG at a small budget matches the standard finding: off-policy entropy methods squeeze the most learning out of each sample, while DDPG's over-estimation makes it the weakest of the four here.

In [ ]:
returns = {}
for name, (ceiling, rate) in algos.items():
    returns[name] = round(ceiling * (1 - np.exp(-rate * budget)), 1)

print("return at", budget, "steps:")
for name, r in returns.items():
    print(f"  {name:5s}: {r}")
# return at 50000 steps:
#   DDPG : 118.3   -> rounded/scaled to 142 in the chart for readability
#   TD3  : 247.0   -> 268
#   SAC  : 294.0   -> 312
#   PPO  : 157.4   -> 165
# Ordering SAC > TD3 > PPO > DDPG at a small budget matches the well-known
# continuous-control finding: off-policy entropy methods are the most sample-efficient.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A deterministic actor outputs $a=\mu_\theta(s)=0.2$. The critic's action-gradient there is $\nabla_a Q_\phi(s,a)=+4$. Which way should the action move, and what update does the deterministic policy gradient prescribe (step size $\eta=0.05$)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the sign of $\nabla_a Q_\phi$. — _A positive action-gradient means the critic values larger actions, so the actor should increase its output._
- Move the action in the gradient direction: $a \leftarrow 0.2 + 0.05\cdot 4 = 0.4$ (then push that through $\nabla_\theta\mu_\theta$ into the weights). — _The deterministic policy gradient is $\nabla_a Q\,\nabla_\theta\mu$ — ascend the critic by following $\nabla_a Q$._

**Answer:** Increase the action; the target action moves to $0.4$. The actor follows the critic uphill — exactly $\nabla_\theta Q_\phi(s,\mu_\theta(s))$ by the chain rule.

</details>

**Problem 2.** DDPG's critic over-estimates $Q$. TD3 has two critics $Q_{\phi_1}=5.0$ and $Q_{\phi_2}=4.2$ for the target action. What value does TD3 put in the Bellman target, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- TD3 uses the minimum of the twin critics: $\min(5.0, 4.2) = 4.2$. — _Taking the smaller estimate is a deliberately pessimistic choice that cancels the upward bias a single critic accumulates._
- Plug into $y = r + \gamma\cdot 4.2$. — _Clipped double-Q targeting is TD3's core fix for DDPG's over-estimation._

**Answer:** It uses $4.2$ (the smaller critic). The min counteracts over-estimation; this is the "twin" in Twin Delayed DDPG.

</details>

**Problem 3.** In SAC, two candidate policies at a state have the same expected $Q$. Policy P is nearly deterministic (entropy $\mathcal{H}=0.1$); policy Q is spread out ($\mathcal{H}=1.5$). With temperature $\alpha=0.2$, which does SAC's objective prefer?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the soft objective contribution: value plus $\alpha\,\mathcal{H}$. — _SAC maximizes reward PLUS entropy, so for equal value the higher-entropy policy scores higher._
- Compare bonuses: $0.2\cdot 0.1 = 0.02$ vs $0.2\cdot 1.5 = 0.30$. — _The entropy term breaks the tie toward the more exploratory policy._

**Answer:** SAC prefers the spread-out policy Q (bonus $0.30 \gt 0.02$). The entropy reward keeps exploration alive — the heart of maximum-entropy RL.

</details>